# Ajuste de Modelos

## General

### Importación de librerías

In [1]:
import warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from itertools import product
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pmdarima import auto_arima
from arch import arch_model
import joblib
warnings.filterwarnings("ignore")

### Importando datos

In [2]:
df = pd.read_csv('entrenamiento.csv', index_col=0)
df_etiquetas = pd.read_csv('entrenamiento_etiquetas.csv', index_col=0)
targets = pd.read_csv('target_pairs.csv')

In [3]:
df.head(2)

,US_Stock_NEM_adj_close,FX_CHFJPY,US_Stock_XOM_adj_close,US_Stock_ALB_adj_close,FX_EURJPY,JPX_Platinum_Standard_Futures_Close,FX_EURAUD,US_Stock_URA_adj_close,FX_CADCHF,US_Stock_OKE_adj_close,...,FX_NOKJPY,FX_EURCHF,FX_NZDUSD,FX_AUDCAD,US_Stock_SCCO_adj_close,US_Stock_HES_adj_close,FX_AUDUSD,US_Stock_WMB_adj_close,FX_NOKUSD,US_Stock_BKR_adj_close
date_id,,,,,,,,,,,,,,,,,,,,,
0,3.416395,4.749668,4.095014,4.783912,4.908602,8.139441,0.432021,2.559767,-0.252477,3.525210,...,2.626315,0.158934,-0.342165,-0.020610,3.553315,3.755879,-0.244121,3.023848,-2.094385,3.257966
1,3.407974,4.746975,4.114464,4.785887,4.907290,8.139441,0.428469,2.561675,-0.250153,3.541362,...,2.631036,0.160315,-0.344013,-0.018001,3.555154,3.787758,-0.244905,3.051091,-2.092689,3.297510


In [4]:
df_etiquetas.head(2)

,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,target_9,...,target_414,target_415,target_416,target_417,target_418,target_419,target_420,target_421,target_422,target_423
date_id,,,,,,,,,,,,,,,,,,,,,
0,0.005948,-0.002851,-0.004675,-0.000639,-0.031852,-0.019452,-0.006729,0.006066,-0.002042,0.003446,...,0.003377,0.021239,-0.005595,0.012846,-0.004628,0.033793,-0.000764,0.038234,0.003548,0.02731
1,0.005783,-0.024118,-0.007052,-0.018955,-0.031852,-0.019452,0.003002,-0.006876,-0.002042,0.021284,...,0.003377,0.021372,-0.001517,0.012846,0.010547,0.030527,-0.000764,0.025021,0.003548,0.02094


In [5]:
targets.head(2)

,target,lag,pair
0,target_0,1,US_Stock_VT_adj_close
1,target_1,1,LME_PB_Close - US_Stock_VT_adj_close


### Obteniendo los targets.

In [6]:
targets['etiquetas'] = [x.split(' - ') for x in targets['pair']]
targets.head(2)

,target,lag,pair,etiquetas
0,target_0,1,US_Stock_VT_adj_close,[US_Stock_VT_adj_close]
1,target_1,1,LME_PB_Close - US_Stock_VT_adj_close,"[LME_PB_Close, US_Stock_VT_adj_close]"


### Eligiendo las columnas que se usarán

In [7]:
columnas_usadas = list(dict.fromkeys(y for x in targets["etiquetas"] for y in x))
columna = 9
nombre_columna = columnas_usadas[columna]
df_filtrado = df[[nombre_columna]]

### Separando train y test

In [8]:
n = int(df_filtrado.shape[0]*0.9)
train = df_filtrado.iloc[:n]
test = df_filtrado.iloc[n:]

In [9]:
print(train.shape)
print(test.shape)

(1764, 1)
(197, 1)


## SARFIMA

### Pesos fraccionales

In [10]:
def pesos_diferenciacion_fraccional(d, n):

    pesos = np.ones(n)

    for k in range(1, n):
        pesos[k] = -pesos[k - 1] * (d - k + 1) / k

    return pesos

### Diferenciación fraccional

In [11]:
def diferenciacion_fraccional(serie, d):

    serie = np.asarray(serie, dtype=float)
    pesos = pesos_diferenciacion_fraccional(d=d, n=len(serie))
    serie_diff = np.convolve(serie, pesos, mode="full")[:len(serie)]

    return serie_diff

### Clase SARFIMA

In [12]:
@dataclass
class ModeloSARFIMA:

    modelo: object
    d_fraccional: float
    historia: np.ndarray
    periodo_estacional: int

    def predict(self, n_periods):

        # Predicción sobre la serie diferenciada
        pred_diff = np.asarray(self.modelo.predict(n_periods=n_periods), dtype=float)

        historia = list(np.asarray(self.historia, dtype=float))
        n_train = len(historia)

        # Se calculan suficientes pesos también para las observaciones futuras
        pesos = pesos_diferenciacion_fraccional(d=self.d_fraccional, n=n_train + n_periods)

        predicciones = []

        # Se reconstruye la serie original
        for z_t in pred_diff:

            t = len(historia)
            acumulado = 0.0

            for k in range(1, t + 1):
                acumulado += (pesos[k] * historia[t - k])

            y_t = z_t - acumulado
            historia.append(y_t)
            predicciones.append(y_t)

        return np.asarray(predicciones)

### Ajuste de SARFIMA

In [13]:
def ajustar_sarfima(y, m=7, d_grid=np.arange(0.05, 0.55, 0.1),
                    max_p=3, max_q=3, max_P=2, max_Q=2):

    mejor_modelo, mejor_aicc, mejor_d = None, np.inf, None

    # Rejilla de modelos
    for d_frac in d_grid:
        try:
            y_frac = diferenciacion_fraccional(y, d_frac)
            modelo = auto_arima(y_frac,

                # Parte ARIMA
                start_p=0, start_q=0, max_p=max_p, max_q=max_q, d=0,

                # Parte estacional
                seasonal=m > 1, m=m, start_P=0, start_Q=0, max_P=max_P, max_Q=max_Q, D=None if m > 1 else 0, max_D=1,

                # Selección automática
                information_criterion="aicc", stepwise=True, error_action="ignore", suppress_warnings=True)

            if np.isfinite(modelo.aicc()) and modelo.aicc() < mejor_aicc:
                mejor_modelo = modelo
                mejor_aicc = modelo.aicc()
                mejor_d = d_frac
        except:
            continue

    # Modelo por default
    if mejor_modelo is None:
        try:
            d_frac = 0.1
            y_frac = diferenciacion_fraccional(y, d_frac)
            mejor_modelo = auto_arima(y_frac, d=0, seasonal=False, max_p=2, max_q=2, stepwise=True, error_action="ignore", suppress_warnings=True)
            mejor_d = d_frac
        except:
            return None

    
    return ModeloSARFIMA(modelo=mejor_modelo, d_fraccional=mejor_d, historia=np.asarray(y), periodo_estacional=m)

## SARIMA

In [14]:
def ajustar_sarima(y, m=7, max_p=3, max_q=3, max_P=2, max_Q=2):

    modelo = auto_arima(y,
                        
        # Parte ARIMA
        start_p=0, start_q=0, max_p=max_p, max_q=max_q, d=None, max_d=2,

        # Parte estacional
        seasonal=(m > 1), m=m, start_P=0, start_Q=0, max_P=max_P, max_Q=max_Q, D=None, max_D=1,

        # Selección automática
        information_criterion="aicc", test="kpss", seasonal_test="ocsb", stepwise=True,
        suppress_warnings=True, error_action="ignore", with_intercept="auto", maxiter=100, trace=False
    )

    return modelo

## GARCH

In [15]:
def ajustar_garch(y, max_ar=4, max_p=3, max_q=3, distribuciones=("normal", "t")):
    mejor_modelo, mejor_bic = None, np.inf

    # Rejilla de modelos
    for ar in range(max_ar + 1):
        for p in range(1, max_p + 1):
            for q in range(1, max_q + 1):
                for dist in distribuciones:
                    try:
                        modelo = arch_model(y, mean="AR" if ar > 0 else "Constant", lags=ar if ar > 0 else None,
                                            vol="GARCH", p=p, q=q, dist=dist)
                        ajuste = modelo.fit(disp="off", show_warning=False)

                        if np.isfinite(ajuste.bic) and ajuste.bic < mejor_bic:
                            mejor_modelo = ajuste
                            mejor_bic = ajuste.bic
                    except:
                        continue

    # Modelo por defaut
    if mejor_modelo is None:
        try:
            mejor_modelo = arch_model(y, mean="Constant", vol="GARCH", p=1, q=1).fit(disp="off", show_warning=False)
        except:
            pass

    return mejor_modelo

## Ajuste de modelos

### Métrica de comparación

In [16]:
def calcular_error(y_real, y_pred):

    y_real = np.asarray(y_real, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return mean_absolute_error(y_real, y_pred)

In [17]:
def seleccionar_mejor_modelo(df, df_test, m=7):
    mejores_modelos, resultados = [], []

    for columna in df.columns:
        y_train = df[columna].dropna().values
        y_test = df_test[columna].dropna().values
        horizonte = len(y_test)
        candidatos = {}

        # GARCH
        modelo = ajustar_garch(y_train)
        if modelo is not None:
            try:
                pred = modelo.forecast(horizon=horizonte, reindex=False).mean.iloc[-1].values
                candidatos["GARCH"] = (modelo, np.sqrt(mean_squared_error(y_test, pred)))
            except:
                pass

        # SARIMA
        modelo = ajustar_sarima(y_train, m=m)
        if modelo is not None:
            try:
                pred = modelo.predict(n_periods=horizonte)
                candidatos["SARIMA"] = (modelo, np.sqrt(mean_squared_error(y_test, pred)))
            except:
                pass

        # SARFIMA
        modelo = ajustar_sarfima(y_train, m=m)
        if modelo is not None:
            try:
                pred = modelo.predict(n_periods=horizonte)
                candidatos["SARFIMA"] = (modelo, np.sqrt(mean_squared_error(y_test, pred)))
            except:
                pass

        # Elección de candidatos
        if candidatos:
            ganador = min(candidatos, key=lambda x: candidatos[x][1])
            modelo, mae = candidatos[ganador]
            mejores_modelos.append(modelo)
            resultados.append({"serie": columna, "modelo": ganador, "MAE": mae})

    return mejores_modelos, pd.DataFrame(resultados)

## Guardando información

### Mejores modelos

In [18]:
mejores_modelos, resultados = seleccionar_mejor_modelo(train, test, m=7)

### Almacenando el mejor modelo

In [19]:
modelo = mejores_modelos[0]
ganador = resultados.iloc[0]["modelo"]
error = resultados.iloc[0]["MAE"]

nombre_archivo = f"modelos/{ganador}_{nombre_columna}.joblib"
joblib.dump(modelo, nombre_archivo)

['modelos/SARFIMA_FX_AUDJPY.joblib']

### CSV con la información de los mejores modelos

In [20]:
mejores_modelos_df = pd.read_csv('mejores_modelos.csv', index_col=0)
mejores_modelos_df.head(2)

,mejor_modelo,error
US_Stock_VT_adj_close,SARIMA,0.042982
LME_PB_Close,SARIMA,0.044852


### Cambiando los datos del CSV

In [21]:
mejores_modelos_df.loc[nombre_columna, 'mejor_modelo'] = ganador
mejores_modelos_df.loc[nombre_columna, 'error'] = error
mejores_modelos_df.loc[nombre_columna]

mejor_modelo    SARFIMA
error           0.03826
Name: FX_AUDJPY, dtype: object

### Guardando nuevamente el CSV

In [22]:
mejores_modelos_df.to_csv('mejores_modelos.csv')